In [1]:
import torch
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import lightning.pytorch as pl
from lightning.pytorch.callbacks import RichProgressBar, Timer, LearningRateFinder
import omegaconf


# Add the prime_torch file to the system path so we can import it
import sys
sys.path.append("/glade/u/home/cobrien/prime/prime_lib/primesw")
from data import SWDataset, SWDataModule
from prime_torch import crps, SWRegressor

In [2]:
torch.set_float32_matmul_precision('medium')

config = '/glade/u/home/cobrien/prime/prime_lib/configs/plasmasheet.yaml'
cfg = omegaconf.OmegaConf.load(
    config
)

# This will take like an hour. Find a way to cache this dataloader
datamodule = SWDataModule(
    target_features = cfg.data.target_features,
    input_features = cfg.data.input_features,
    position_features = cfg.data.position_features,
    interp_flags = cfg.data.interp_flags,
    region = cfg.data.region,
    cuts = cfg.data.cuts,
    cadence = cfg.data.cadence,
    interpolate = cfg.data.interpolate,
    window = cfg.data.window,
    stride = cfg.data.stride,
    interp_frac = cfg.data.interp_frac,
    trn_bounds = cfg.data.trn_bounds,
    val_bounds = cfg.data.val_bounds,
    tst_bounds = cfg.data.tst_bounds,
    batch_size = cfg.opt.batch_size,
    num_workers = cfg.opt.num_workers,
    datastore = cfg.data.datastore,
    in_key = cfg.data.in_key,
    tar_key = cfg.data.tar_key,
)

In [3]:
model = SWRegressor.load_from_checkpoint(
    "/glade/u/home/cobrien/data/prime/tensorboard_logs/pstesting/version_9/checkpoints/epoch=99-step=3900.ckpt"
)

/glade/work/cobrien/conda-envs/pt212gpu_conda/lib/python3.11/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
x = torch.from_numpy(
    np.random.normal(size = (1, cfg.data.window, len(datamodule.input_features))).astype(np.float32)
)

pos = torch.from_numpy(
    np.random.normal(size = (1, len(datamodule.position_features))).astype(np.float32)
)

y_hat = model.forward(
    x, 
    pos
)

In [5]:
out, h = model.encoder(x)
out.shape

torch.Size([1, 200, 128])

In [6]:
y_hat.shape

torch.Size([1, 4])

In [7]:
testmodel = SWRegressor(
    optimizer = cfg.opt.optimizer,
    lr = cfg.opt.lr,
    lr_scheduler = cfg.opt.lr_scheduler,
    patience = cfg.opt.patience,
    factor = cfg.opt.factor,
    weight_decay = cfg.opt.weight_decay,
    total_iters = cfg.opt.total_iters,
    in_dim = len(cfg.data.input_features),
    tar_dim = len(cfg.data.target_features),
    pos_dim = len(cfg.data.position_features),
    in_norm = datamodule.input_normalizations,
    tar_norm = datamodule.target_normalizations,
    pos_norm = datamodule.position_normalizations,
    window = cfg.data.window,
    stride = cfg.data.stride,
    interp_frac = cfg.data.interp_frac,
    decoder_type = cfg.model.decoder_type,
    encoder_type = cfg.model.encoder_type,
    decoder_hidden_layers = cfg.model.decoder_hidden_layers,
    encoder_hidden_dim = cfg.model.encoder_hidden_dim,
    encoder_num_layers=cfg.model.encoder_num_layers,
    p_drop = cfg.model.p_drop,
    pos_encoding_size=cfg.model.pos_encoding_size,
    loss=cfg.opt.loss,
)